<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/Evaluating-Quantized-LM-Using-LLM-Judge/blob/main/%5Cnotebooks%5CLLM_Judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Quantized Language Models Using an LLM Judge

This notebook evaluates Qwen2.5-1.5B-Instruct in three configurations:

- FP16
- 8-bit LLM.int8()
- 4-bit NF4

The experiments compare model accuracy, GPU memory usage, latency, throughput, formatting failures, and reasoning behavior on the GSM8K dataset.

# Quantized Language Model Evaluation

This notebook runs the experiments for the project
“Evaluating Quantized Language Models Using an LLM Judge.”

It will:

- Clone the GitHub repository.
- Verify the project files.
- Install the required packages.
- Confirm the available GPU.
- Run Qwen2.5-1.5B-Instruct in FP16, 8-bit, and 4-bit NF4.
- Save generated responses and GPU performance metrics.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = (
    "https://github.com/"
    "Ramprasath-Meganathan/"
    "Evaluating-Quantized-LM-Using-LLM-Judge.git"
)

PROJECT_DIR = Path(
    "/content/Evaluating-Quantized-LM-Using-LLM-Judge"
)

SRC_DIR = PROJECT_DIR / "src"

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )
else:
    print("Repository already exists.")

required_files = [
    "README.md",
    "requirements.txt",
    "src/config.py",
    "src/model_utils.py",
    "src/generation_utils.py",
    "src/run_generation.py",
]

for filename in required_files:
    file_path = PROJECT_DIR / filename

    assert file_path.is_file(), (
        f"Missing project file: {filename}"
    )

os.chdir(PROJECT_DIR)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

(PROJECT_DIR / "data").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)

print("Project directory:", os.getcwd())
print("Repository setup completed.")

## Install Dependencies

The project uses:

- PyTorch for GPU inference.
- Transformers for loading the language model.
- Accelerate for device placement.
- bitsandbytes for 8-bit and 4-bit quantization.
- Datasets for loading and processing GSM8K.
- NumPy and pandas for data processing and analysis.

In [ ]:
!pip install -r requirements.txt

## Verify GPU Availability

The experiments require an NVIDIA GPU.

The GPU name and available memory are recorded because inference speed and memory usage depend on the hardware used.

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    raise RuntimeError("CUDA GPU is not available.")

## Prepare the Dataset

The generation script expects the cleaned GSM8K dataset at:

`data/gsm8k_clean.jsonl`

Each record should contain:

- `problem_id`
- `question`
- `reference_answer`
- `reasoning_step_count`

The same questions must be used for FP16, 8-bit, and 4-bit experiments.

In [ ]:
from pathlib import Path
import json

data_path = Path("data/gsm8k_clean.jsonl")

if not data_path.exists():
    raise FileNotFoundError(
        "data/gsm8k_clean.jsonl was not found. "
        "Prepare the dataset before running generation."
    )

with open(data_path, "r", encoding="utf-8") as file:
    first_record = json.loads(file.readline())

print("First dataset record:")
print(first_record)

## Experimental Settings

All model configurations use the same:

- Model family.
- Dataset.
- Problem order.
- Prompt.
- Maximum number of generated tokens.
- Decoding strategy.
- Batch size.
- Random seed.

The only intended experimental difference is the model precision.

In [ ]:
from config import (
    MODEL_NAME,
    SEED,
    MAX_NEW_TOKENS,
    BATCH_SIZE,
)

print("Model:", MODEL_NAME)
print("Seed:", SEED)
print("Maximum new tokens:", MAX_NEW_TOKENS)
print("Batch size:", BATCH_SIZE)

## FP16 Baseline Experiment

The FP16 model serves as the baseline.

Its generated responses and hardware measurements will be used for comparison with the 8-bit and 4-bit configurations.

In [ ]:
!python src/run_generation.py \
  --precision fp16 \
  --input data/gsm8k_clean.jsonl \
  --output results/fp16_results.jsonl

## 8-bit Quantization Experiment

This experiment evaluates the same Qwen2.5-1.5B-Instruct model using 8-bit LLM.int8() quantization.

The dataset, prompt, decoding settings, and problem order remain unchanged.

In [ ]:
!python src/run_generation.py \
  --precision int8 \
  --input data/gsm8k_clean.jsonl \
  --output results/int8_results.jsonl

## 4-bit NF4 Quantization Experiment

This experiment evaluates the same model using 4-bit NF4 quantization.

The results will show whether stronger weight compression affects mathematical accuracy, reasoning behavior, memory usage, and inference speed.

In [ ]:
!python src/run_generation.py \
  --precision nf4 \
  --input data/gsm8k_clean.jsonl \
  --output results/nf4_results.jsonl

## Verify Generated Results

The experiments should produce:

- `fp16_results.jsonl`
- `int8_results.jsonl`
- `nf4_results.jsonl`
- `gpu_metrics.csv`

The JSONL files contain the generated model responses.

The metrics file contains GPU memory, latency, throughput, and experiment metadata.

In [ ]:
from pathlib import Path

results_dir = Path("results")

for file_path in sorted(results_dir.iterdir()):
    print(file_path, "-", file_path.stat().st_size, "bytes")

In [ ]:
import json

result_files = [
    "results/fp16_results.jsonl",
    "results/int8_results.jsonl",
    "results/nf4_results.jsonl",
]

for result_file in result_files:
    path = Path(result_file)

    if path.exists():
        with open(path, "r", encoding="utf-8") as file:
            first_result = json.loads(file.readline())

        print("\nFile:", result_file)
        print("Precision:", first_result["precision"])
        print("Problem ID:", first_result["problem_id"])
        print("Response:")
        print(first_result["response"])
    else:
        print("Missing:", result_file)

## Inspect GPU Metrics

The GPU metrics file contains measurements for each precision, including:

- Peak GPU memory.
- Total generation time.
- Number of generated tokens.
- Tokens per second.
- GPU name.

In [ ]:
import pandas as pd

metrics_path = Path("results/gpu_metrics.csv")

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df)
else:
    print("gpu_metrics.csv was not found.")

## Next Evaluation Stage

The generated responses will next be processed to:

- Extract final numerical answers.
- Compare predictions with GSM8K reference answers.
- Calculate exact-match accuracy.
- Measure formatting failures.
- Analyze accuracy by reasoning difficulty.
- Compare FP16, 8-bit, and 4-bit behavior.
- Evaluate the LLM judge.